# California Wildfire PM2.5 Pollution on Prison and Detention Facilities

## Part 3: Pre-Analysis and Finalizing Features Before Inputting into Model

**EDA, One Hot Encoding and Observing Feature Dependency**:
1. EDA
2. One Hot Encoding Facility and Sensor Types
3. Correlation Matrix
2. Clustering

In [26]:
import os
import re
import ee
import geemap
import json
import datetime 
import requests
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd 
import seaborn as sns
import folium
import matplotlib.pyplot as plt

import geopandas as gpd

from functools import reduce
from datetime import datetime
from dotenv import load_dotenv, find_dotenv
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Obtaining Path files to the data in data folder

directory_name = os.getcwd()
directory_path = Path(directory_name)

# Loading final engineering dataframe: 
final_feature_engineered_data = pd.read_pickle(directory_path.joinpath('data/final_feature_engineering_features.pkl'))

## EDA on Feature Engineered DataFrame

In [27]:
final_feature_engineered_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5237 entries, 0 to 5236
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Institution Name                5237 non-null   object 
 1   Date                            5237 non-null   object 
 2   min_distance_km                 5237 non-null   float64
 3   max_magnitude_score             5237 non-null   float64
 4   total_magnitude                 5237 non-null   float64
 5   max_acres                       5237 non-null   float32
 6   total_acres                     5237 non-null   float32
 7   intersecting_fires              5237 non-null   float64
 8   number_of_fires_at_10km         5237 non-null   float64
 9   number_of_fires_at_50km         5237 non-null   float64
 10  number_of_fires_at_100km        5237 non-null   float64
 11  firms_50km_max_frp              5237 non-null   float64
 12  firms_50km_total_frp            52

In [3]:
# Count of NA values in dataframe
final_feature_engineered_data.isna().sum()

Institution Name                     0
Date                                 0
min_distance_km                      0
max_magnitude_score                  0
total_magnitude                      0
max_acres                            0
total_acres                          0
intersecting_fires                   0
number_of_fires_at_10km              0
number_of_fires_at_50km              0
number_of_fires_at_100km             0
firms_50km_max_frp                   0
firms_50km_total_frp                 0
firms_50km_total_thermal_score       0
firms_50km_fire_day_count            0
firms_50km_pixel_count               0
firms_50km_night_fire_ratio          0
fire_50km_elevation_mean          2018
fire_50km_elevation_max           2018
facility_elevation                1983
5km_buffer_elevation              1983
valley_index                      1983
max_pm_conc                       1306
avg_pm_conc                       1306
max_aqi                           1306
avg_monitor_dist         

In [4]:
# Count of 0 values in dataframe
(final_feature_engineered_data == 0).sum()

Institution Name                     0
Date                                 0
min_distance_km                   2541
max_magnitude_score               2536
total_magnitude                   2536
max_acres                         2536
total_acres                       2536
intersecting_fires                5231
number_of_fires_at_10km           5038
number_of_fires_at_50km           3611
number_of_fires_at_100km          2834
firms_50km_max_frp                1983
firms_50km_total_frp              1983
firms_50km_total_thermal_score    1983
firms_50km_fire_day_count         1983
firms_50km_pixel_count            1983
firms_50km_night_fire_ratio       4646
fire_50km_elevation_mean             0
fire_50km_elevation_max              0
facility_elevation                   0
5km_buffer_elevation                 0
valley_index                         0
max_pm_conc                          0
avg_pm_conc                          0
max_aqi                              0
avg_monitor_dist         

In [5]:
# Removing columns that are missing 80% non-structural data
finalizing_features = final_feature_engineered_data.copy()
finalizing_features.drop(columns='intersecting_fires', inplace=True)

In [20]:
# Inspecting NA PM 2.5 date values
missing_pm_values_df = finalizing_features[finalizing_features['avg_pm_conc'].isna()]
missing_pm_values_sorted = missing_pm_values_df.sort_values(by="Date", ascending=True)
missing_pm_values_sorted['Date'].value_counts().head(40)

Date
2020-12    79
2020-11    79
2020-10    79
2021-11    77
2021-10    77
2021-12    77
2022-10    72
2022-12    72
2022-11    72
2024-12    71
2023-10    70
2023-11    70
2023-12    70
2024-10    70
2024-11    70
2020-09    12
2020-02     7
2020-01     7
2021-01     7
2020-05     7
2020-04     7
2021-05     7
2021-04     7
2020-03     7
2020-06     7
2020-07     7
2020-08     7
2021-02     7
2021-03     7
2022-07     6
2021-07     6
2022-03     6
2022-06     6
2022-05     6
2021-06     6
2022-01     6
2022-02     6
2022-04     6
2022-08     4
2021-09     3
Name: count, dtype: int64

In [19]:
# Inspecting facilities with missing PM 2.5 data
missing_pm_values_sorted['Institution Name'].value_counts().head(40)

Institution Name
Intermountain #22                            64
Ironwood State Prison (ISP)                  35
Chuckawalla Valley State Prison (CVSP)       31
High Desert State Prison (HDSP)              30
Antelope #25                                 30
California Correctional Center (CCC)         30
Parlin Fork #6                               21
Eel River #31                                17
Sugar Pine #9                                16
Deadwood #23                                 16
Konocti #27                                  16
California Correctional Institution (CCI)    16
Calipatria State Prison (CAL)                16
California City Correctional Facility        16
Washington Ridge #44                         15
Cuesta #24                                   15
Vallecito #1                                 15
Adelanto ICE Processing Center               15
Holton #16                                   15
Ventura Training Center (VTC)                15
Valley State Prison (VS

## Fixing Missing Elevation and PM2.5 Data

In [8]:
# Setting each facility_elevation, buffer, and valley index to each facility
missing_fac_elevation_data = finalizing_features[finalizing_features['facility_elevation'].isna()]
nonmissing_fac_elevation_data = finalizing_features[(finalizing_features['facility_elevation'].notna())].copy()
single_fac_elevation_data = nonmissing_fac_elevation_data.drop_duplicates(subset='Institution Name')

for index, row in missing_fac_elevation_data.iterrows():
    filter_by_fac = single_fac_elevation_data[single_fac_elevation_data['Institution Name'] == row['Institution Name']]
    finalizing_features.iloc[index, 18:21]= filter_by_fac[['facility_elevation', '5km_buffer_elevation', 'valley_index']]

In [33]:
# Setting aside missing pm values to be inputted in model after it is established.
# Missing pm values resulted in no EPA or PurpleAir data existing at the specified time frame and/or location.
missing_pm_values_indices = missing_pm_values_df.index

finalizing_features_all_pm = finalizing_features.drop(missing_pm_values_indices)
finalizing_features_all_pm.shape

finalizing_features_all_pm.shape

(3931, 32)

## One Hot Encoding